In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)
emb = embedding_factory(
    "openai",
    model="text-embedding-3-small",
    client=client
)

### RAG 메트릭

#### 검색 - Context Precision

$$
\text{Context Precision@K}
=
\sum_{k=1}^{K}
\left(
\text{Precision@k} \times v_k
\right)
$$

상위 $K$개 검색 결과 중 **관련 청크가 얼마나 상위에 위치했는지** 평가

$$
\text{Precision@k}
=
\frac{TP@k}{TP@k + FP@k}
$$

- $K$ = retrieved contexts 개수
- $v_k \in \{0,1\}$ = $k$번째 청크의 관련 여부

#### 예시

질문: **"에펠탑은 어디 있어?"**

- [**파리 에펠탑 청크**, 베를린 브란덴부르크 문 청크] → **1.0**
  - 관련 청크가 1등이면 뒤에 쓰레기가 있어도 점수 유지

- [베를린 청크, **파리 에펠탑 청크**] → **0.5**
  - P@2 = 1/2, 관련 청크 1개

#### Context Precision 구현 방식

| **클래스** | **관련 여부 판단 기준** | **언제** |
| :--- | :--- | :--- |
| `ContextPrecision` | LLM이 각 청크를 **reference**와 비교 | 정답 답변이 있을 때 |
| `ContextUtilization` | LLM이 각 청크를 생성된 **response**와 비교 | 정답 없이 응답만 있을 때 |
| `NonLLMContextPrecisionWithReference` | retrieved vs reference_contexts를 Levenshtein 등 문자열 유사도로 비교 (`pip install rapidfuzz`) | 정답 컨텍스트가 있고 LLM 비용을 아낄 때 |
| `IDBasedContextPrecision` | 문서 ID 일치 | 문서에 고유 ID 체계가 있을 때. 가장 빠르고 결정적 |

#### ID-Based Precision

$$
\text{ID-Based Precision}
=
\frac{\text{retrieved ID 중 reference ID에 포함된 수}}
{\text{retrieved ID 전체 수}}
$$

**예시**

- `retrieved = [doc_1, doc_2, doc_3, doc_4]`
- `reference = [doc_1, doc_4, doc_5, doc_6]`

  공통 ID는 `doc_1`, `doc_4` → **2개**

  $$
  \text{ID-Based Precision}
  =
  \frac{2}{4}
  =
  \mathbf{0.5}
  $$

  > ID 기반 Precision은 **순위 가중치 없이 단순 비율**로 계산

In [ ]:
# 코드
from ragas.metrics.collections import ContextPrecision, ContextUtilization
from ragas import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision

# ContextPrecision
cp = ContextPrecision(llm=llm)
r = await cp.ascore(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris.",
                        "The Brandenburg Gate is located in Berlin."],
)

# ContextUtilization
cu = ContextUtilization(llm=llm)
# reference 대신 response
await cu.ascore(user_input=..., response=..., retrieved_contexts=[...])

# SingleTurnSample
s = SingleTurnSample(retrieved_context_ids=["doc_1","doc_2","doc_3","doc_4"],
                     reference_context_ids=["doc_1","doc_4","doc_5","doc_6"])

# IDBasedContextPrecision
await IDBasedContextPrecision().single_turn_ascore(s)   # 0.5

#### 검색 - Context Recall

**목적:** 필요한 정보를 **빠뜨리지 않고** 가져왔는지 평가
**핵심 질문:** "놓친 정보가 없나?"

- LLM Reference 필수
- 검색 결과가 정답에 필요한 정보를 얼마나 포함하는지 평가
- 점수 범위: **0~1 ↑**

> 비교 대상이 되는 **정답이 반드시 필요**함.
> LLM 버전은 `reference_contexts`를 일일이 라벨링하기 어려우므로 **reference(정답 답변)를 기준**으로 사용

**평가 과정**

1. `reference`를 **claim 단위**로 쪼갬
2. 각 claim이 `retrieved_contexts`로 **뒷받침되는지** 판정
3. 뒷받침되는 claim의 **비율 계산**

$$
\text{Context Recall}
=
\frac{\text{retrieved context로 뒷받침되는 reference claim 수}}
{\text{reference 전체 claim 수}}
$$

$$
\text{NonLLM Context Recall}
=
\frac{\text{관련 있다고 판정된 retrieved context 수}}
{\text{reference\_contexts 전체 수}}
$$

$$
\text{ID-Based Recall}
=
\frac{\text{reference ID 중 retrieved에 포함된 수}}
{\text{reference ID 전체 수}}
$$

**예시**

- **NonLLM**

  `retrieved = ["Paris is the capital of France."]`

  `reference_contexts = [그 문장, "에펠탑은 파리의 랜드마크"]`

  관련 retrieved context = **1개**, reference context = **2개**

  $$
  \text{NonLLM Recall}
  =
  \frac{1}{2}
  =
  \mathbf{0.5}
  $$

- **ID-Based**

  `retrieved = [doc_1, doc_2, doc_3]`

  `reference = [doc_1, doc_4, doc_5, doc_6]`

  공통 ID = `doc_1` → **1개**

  $$
  \text{ID-Based Recall}
  =
  \frac{1}{4}
  =
  \mathbf{0.25}
  $$

> **Precision**은 "가져온 것 중 얼마나 관련 있나"
> **Recall**은 "필요한 것 중 얼마나 가져왔나"

In [ ]:
# 코드
from ragas.metrics.collections import ContextRecall

r = await ContextRecall(llm=llm).ascore(
    user_input="Where is the Eiffel Tower located?",
    retrieved_contexts=["Paris is the capital of France."],
    reference="The Eiffel Tower is located in Paris.",
)

#### 검색 - Context Entities Recall

**목적:** reference에 나오는 **개체(인명·지명·연도 등)** 중 얼마나 많은 개체가 검색된 컨텍스트에 포함되어 있는지 평가
**핵심 질문:** "정답에 필요한 고유명사를 얼마나 빠뜨리지 않고 가져왔나?"

- LLM Reference 필수
- 점수 범위: **0~1 ↑**
- 여행 헬프데스크, 역사 QA처럼 **사실·고유명사가 중요한 경우** 유용

$$
\text{Context Entity Recall}
=
\frac{|RCE \cap RE|}
{|RE|}
$$

- $RE$ = reference에 포함된 엔티티
- $RCE$ = retrieved context에 포함된 엔티티

**예시**

타지마할 설명에 대한 reference 엔티티:

`RE = [Taj Mahal, Yamuna, Agra, 1631, Shah Jahan, Mumtaz Mahal]`

→ 총 **6개**

- **컨텍스트 1**

  `RCE = [Taj Mahal, Agra, Shah Jahan, Mumtaz Mahal, India]`

  reference와 겹치는 엔티티 = **4개**

  $$
  \text{Context Entity Recall}
  =
  \frac{4}{6}
  =
  \mathbf{0.667}
  $$

- **컨텍스트 2**

  `RCE = [Taj Mahal, UNESCO, India]`

  reference와 겹치는 엔티티 = **1개**

  $$
  \text{Context Entity Recall}
  =
  \frac{1}{6}
  =
  \mathbf{0.167}
  $$

> 같은 문서셋이라면 **컨텍스트 1을 검색한 방식이 엔티티 중심 과제에 더 적합**함.

In [ ]:
# 코드
from ragas.metrics.collections import ContextEntityRecall

await ContextEntityRecall(llm=llm).ascore(
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."],
)

#### 답변 품질 - Noise Sensitivity

**목적:** 관련 있는(또는 관련 없는) 문서를 가져왔을 때 시스템이 **얼마나 자주 틀린 주장을 하는지** 평가
**핵심 질문:** "검색된 정보의 노이즈 때문에 잘못된 주장을 얼마나 했나?"

- LLM Reference 필수
- 점수 범위: **0~1 ↓**
- **낮을수록 좋음**
- `user_input`, `reference`, `response`, `retrieved_contexts` 4개를 모두 사용
- RAGChecker에서 유래한 개념

$$
\text{Noise Sensitivity}
=
\frac{\text{response 중 틀린 claim 수}}
{\text{response 전체 claim 수}}
$$

**평가 과정**

1. `ground truth`를 추론할 수 있는 **relevant context**를 식별
2. 답변의 각 claim이 relevant context에서 **추론 가능한지** 확인
3. `ground truth`로 뒷받침되지 않는 claim을 **틀림**으로 판정
4. 틀린 claim의 비율 계산

**예시**

질문: **"LIC(인도생명보험공사)는 뭐로 유명해?"**

관련 컨텍스트:

- 인도 최대 보험사
- 대규모 투자 포트폴리오

무관한 컨텍스트:

- 인도 경제 성장에 관한 내용

답변:

1. 인도 최대 보험사
2. 대규모 투자 포트폴리오
3. **국가 금융 안정에 기여**

③은 **ground truth에 없는 주장**이므로 틀린 claim으로 판정

→ 틀린 claim **1개 / 전체 3개**

$$
\text{Noise Sensitivity}
=
\frac{1}{3}
=
\mathbf{0.333}
$$

> 핵심은 **노이즈가 포함된 컨텍스트를 보고 모델이 잘못된 주장을 생성하는 정도**를 측정하는 것.

In [ ]:
# 코드
from ragas.metrics.collections import NoiseSensitivity

scorer = NoiseSensitivity(llm=llm)
scorer_irr = NoiseSensitivity(
    llm=llm,
    mode='irrelevant'
)
await scorer.ascore(
    user_input="",
    response="",
    reference="",
    retrieved_contexts=[],
)

#### 답변 품질 - Answer Relevancy

**목적:** 답변이 **질문 의도에 딱 맞게** 답했는지 평가
**핵심 질문:** "질문에서 요구한 내용을 빠짐없이, 불필요한 내용 없이 답했나?"

- LLM Reference 불필요
- 사실 여부는 평가하지 않음
- 불완전하거나 질문과 관계없는 내용이 섞이면 감점
- 아이디어: **좋은 답변이라면 답변만 보고 원래 질문을 역으로 복원할 수 있다**는 가정

**평가 과정**

1. LLM이 `response`로부터 가상의 질문 **N개** 생성
   - 기본값: **3개**
   - `strictness`로 조절
2. 원래 질문의 임베딩 $E_o$와 생성된 질문의 임베딩 $E_{g_i}$ 사이의 **코사인 유사도** 계산
3. N개 유사도의 **평균** 계산

$$
\text{Answer Relevancy}
=
\frac{1}{N}
\sum_{i=1}^{N}
\frac{E_{g_i} \cdot E_o}
{\|E_{g_i}\|\|E_o\|}
$$

- $E_o$ = 원래 질문의 임베딩
- $E_{g_i}$ = $i$번째 생성 질문의 임베딩
- $N$ = 생성하는 가상의 질문 개수

**예시**

질문: **"프랑스는 어디 있고 수도는?"**

- **낮은 점수**

  답변: "프랑스는 서유럽에 있다."

  → 생성 질문이 대부분 **"프랑스는 어디에 있는가?"**에 가까움

  → 원래 질문의 **수도에 대한 의도**를 반영하지 못함

- **높은 점수**

  답변: "프랑스는 서유럽에 있고 수도는 파리다."

  → 생성 질문이 **프랑스의 위치 + 수도**라는 원래 질문의 의도를 잘 복원함

> **주의:** 코사인 유사도는 이론적으로 **-1~1** 범위이지만, 일반적인 텍스트 임베딩에서는 대체로 **0~1에 가까운 값**이 나타남.

In [ ]:
# 코드
from ragas.metrics.collections import AnswerRelevancy

scorer = AnswerRelevancy(llm=llm, embeddings=emb)
await scorer.ascore(
    user_input = "When was the first super bowl?",
    response="The first superbowl was held on Jan 15, 1967"
)

#### 답변 품질 - Faithfulness

**목적:** 응답이 **검색된 컨텍스트와 사실적으로 일치하는지** 평가
**핵심 질문:** "답변의 모든 claim이 컨텍스트로 뒷받침되는가?"

- LLM Reference 불필요
- 할루시네이션 측정의 대표적인 지표
- 점수 범위: **0~1 ↑**
- 모든 claim이 컨텍스트에 의해 뒷받침될수록 높은 점수

**평가 과정**

1. 응답을 개별 주장(**claim**)으로 분해
2. 각 claim이 컨텍스트에서 **추론 가능한지** 판정
3. 뒷받침되는 claim의 **비율 계산**

$$
\text{Faithfulness}
=
\frac{\text{컨텍스트로 뒷받침되는 응답 claim 수}}
{\text{응답 전체 claim 수}}
$$

**예시**

컨텍스트:

> 아인슈타인(1879년 3월 14일생)은 독일 태생 이론물리학자...

답변:

> "아인슈타인은 독일에서 **1879년 3월 20일**에 태어났다."

답변을 2개의 claim으로 분해:

1. 독일 출생 → **Yes**
2. 1879년 3월 20일 출생 → **No**

→ 뒷받침되는 claim **1개 / 전체 2개**

$$
\text{Faithfulness}
=
\frac{1}{2}
=
\mathbf{0.5}
$$

#### HHEM 변형

기본 Faithfulness에서는 **2단계(claim ↔ 컨텍스트 검증)**를 LLM이 수행하지만, HHEM 변형에서는 이를 Vectara의 **HHEM-2.1-Open**으로 대체

- T5 기반 환각 탐지 분류기
- 무료·소형·오픈소스
- 프로덕션 환경에서 LLM 비용 절감에 유리
- 기본 CPU 사용
- `batch_size=10`